In [ ]:
# ============================================================
# MODELO CHURN - ENFOQUE C 
# ============================================================

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

from imblearn.over_sampling import SMOTE

# ============================================================
# 1. CARGAR DATASETS
# ============================================================

clientes = pd.read_csv("data/processed/df_clientes.csv")
facturacion = pd.read_csv("data/processed/facturacion_limpio.csv")
churn = pd.read_csv("data/processed/df_churn.csv")

# ============================================================
# 2. ARREGLAR TIPOS
# ============================================================

clientes["cliente_id"] = clientes["cliente_id"].astype(str)
facturacion["cliente_id"] = facturacion["cliente_id"].astype(str)
churn["cliente_id"] = churn["cliente_id"].astype(str)

facturacion["fecha"] = pd.to_datetime(facturacion["fecha"], format="mixed", dayfirst=True, errors="coerce")
churn["fecha"] = pd.to_datetime(churn["fecha"], format="mixed", dayfirst=True, errors="coerce")

# ============================================================
# 3. COLUMNAS NECESARIAS
# ============================================================

clientes_modelo = clientes[[
    "cliente_id",
    "edad",
    "antiguedad_meses",
    "ingreso_estimado",
    "num_lineas"
]].drop_duplicates()

facturacion_modelo = facturacion[[
    "cliente_id",
    "fecha",
    "cargo_base",
    "consumo_extra",
    "descuento_aplicado",
    "importe_total",
    "dias_retraso_pago",
    "impago_flag",
    "variacion_consumo_pct",
    "stress_calidad_lag",
    "incidencia_masiva_lag"
]].drop_duplicates()

churn_modelo = churn[[
    "cliente_id",
    "fecha",
    "churn"
]].drop_duplicates()

# ============================================================
# 4. MERGE ENFOQUE C
# ============================================================

df_model = churn_modelo.merge(
    facturacion_modelo,
    on=["cliente_id", "fecha"],
    how="inner"
)

df_model = df_model.merge(
    clientes_modelo,
    on="cliente_id",
    how="inner"
)

# ============================================================
# 5. VARIABLES DEL MODELO
# ============================================================

variables = [
    "cargo_base",
    "consumo_extra",
    "descuento_aplicado",
    "importe_total",
    "dias_retraso_pago",
    "impago_flag",
    "variacion_consumo_pct",
    "stress_calidad_lag",
    "incidencia_masiva_lag",
    "edad",
    "antiguedad_meses",
    "ingreso_estimado",
    "num_lineas"
]

# ============================================================
# 6. LIMPIEZA FINAL
# ============================================================

df_model = df_model.dropna(subset=variables + ["churn"])
df_model = df_model.drop_duplicates()
df_model["churn"] = df_model["churn"].astype(int)

print("="*70)
print("CHECK DATASET ORIGINAL")
print("="*70)

print("\nShape:")
print(df_model.shape)

print("\nDistribución churn:")
print(df_model["churn"].value_counts())

print("\nPorcentaje churn:")
print(df_model["churn"].value_counts(normalize=True) * 100)

# ============================================================
# 7. X / y
# ============================================================

X = df_model[variables]
y = df_model["churn"]

# ============================================================
# 8. TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\nDistribución TRAIN antes de SMOTE:")
print(y_train.value_counts())

print("\nDistribución TEST original:")
print(y_test.value_counts())

# ============================================================
# 9. SMOTE SOLO EN TRAIN
# ============================================================

smote = SMOTE(
    sampling_strategy=0.8,
    random_state=42,
    k_neighbors=3
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nDistribución TRAIN después de SMOTE:")
print(y_train_smote.value_counts())

# ============================================================
# 10. MODELO RANDOM FOREST
# ============================================================

modelo = RandomForestClassifier(
    n_estimators=200,
    max_depth=7,
    min_samples_split=30,
    min_samples_leaf=15,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

modelo.fit(X_train_smote, y_train_smote)

# ============================================================
# 11. PREDICCIONES
# ============================================================

y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

# ============================================================
# 12. RESULTADOS
# ============================================================

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\n" + "="*70)
print("RESULTADOS MODELO ENFOQUE C CON SMOTE")
print("="*70)

print("\nAccuracy:")
print(round(accuracy * 100, 2), "%")

print("\nROC AUC:")
print(round(roc_auc, 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ============================================================
# 13. IMPORTANCIA VARIABLES
# ============================================================

importancia = pd.DataFrame({
    "variable": variables,
    "importancia": modelo.feature_importances_
}).sort_values(by="importancia", ascending=False)

print("\n" + "="*70)
print("IMPORTANCIA VARIABLES")
print("="*70)

print(importancia)

CHECK DATASET ORIGINAL

Shape:
(26931, 16)

Distribución churn:
churn
0    26799
1      132
Name: count, dtype: int64

Porcentaje churn:
churn
0    99.509859
1     0.490141
Name: proportion, dtype: float64

Distribución TRAIN antes de SMOTE:
churn
0    20099
1       99
Name: count, dtype: int64

Distribución TEST original:
churn
0    6700
1      33
Name: count, dtype: int64

Distribución TRAIN después de SMOTE:
churn
0    20099
1    16079
Name: count, dtype: int64

RESULTADOS MODELO ENFOQUE C CON SMOTE

Accuracy:
91.94 %

ROC AUC:
0.6659

Matriz de confusión:
[[6185  515]
 [  28    5]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96      6700
           1       0.01      0.15      0.02        33

    accuracy                           0.92      6733
   macro avg       0.50      0.54      0.49      6733
weighted avg       0.99      0.92      0.95      6733


IMPORTANCIA VARIABLES
                 variable  import

In [43]:
print("\nAccuracy:")
print(round(accuracy * 100, 2), "%")


Accuracy:
91.94 %
